# Colab driver: quantized and binary ViTs for cervical TZ classification

Runs the whole protocol on a Colab GPU. The repository lives on Google Drive (so `runs/`, `quant/`, `analysis/` survive
disconnects); images are cached on the VM's local disk. Every step is resumable — rerun a cell after a disconnect.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
REPO = '/content/drive/MyDrive/cervical-tz-quant-vit'
if not os.path.exists(REPO):
    !git clone https://github.com/mach-12/cervical-tz-quant-vit.git "{REPO}"
else:
    !git -C "{REPO}" pull --ff-only
os.chdir(REPO)
os.environ['TZQ_DATA'] = '/content/data'   # local disk: fast image reads

In [ ]:
!pip -q install "timm>=1.0.20" "transformers>=4.56" fvcore onnx onnxruntime onnxscript gdown

Optional: a Hugging Face token (Colab secret `HF_TOKEN`) is needed only for MedSigLIP (P3), after accepting its licence.

In [ ]:
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as e:
    print('no HF_TOKEN secret; MedSigLIP (P3) will be skipped:', e)

## Data (first session: download ~4 GB and build the cache; later sessions: restore the cache from Drive)

In [ ]:
# The 512 px image cache (~0.4 GB) is kept on Drive, so later sessions skip the 4 GB download.
CACHE_TAR = '/content/drive/MyDrive/tzq_cache/sq512.tar'
if os.path.exists(CACHE_TAR) and not os.path.exists('/content/data/sq512'):
    !mkdir -p /content/data && tar -xf "{CACHE_TAR}" -C /content/data
!python scripts/prepare_data.py
if not os.path.exists(CACHE_TAR):
    !mkdir -p /content/drive/MyDrive/tzq_cache && tar -cf "{CACHE_TAR}" -C /content/data sq512

## E1: frozen-feature probes

In [ ]:
probes = 'hsv dinov2_b dinov3_b' + (' medsiglip' if os.environ.get('HF_TOKEN') else '')
!python scripts/e1_probe.py --probe {probes} --workers 4

## E2: fine-tuning (R3 first; LR search on seed 0, then seeds 1–2, then Test once)

In [ ]:
!python scripts/e2_finetune.py --model dinov3_b --workers 4

In [ ]:
!python scripts/e2_finetune.py --model deit_s --workers 4

In [ ]:
!python scripts/e2_finetune.py --model tinyvit_11m --workers 4

In [ ]:
!python scripts/e2_finetune.py --model mnv4_m --workers 4

In [ ]:
!python scripts/e2_finetune.py --model resnet50 --workers 4

## E6: modification on top of R3 (needs all R3 seeds)
Control (`dinov3_b_cont`, softmax head) and ordinal head (`dinov3_b_ord`), each continued from the R3 checkpoint of the same seed. Run before E3: the E3a/E3b cells then also quantize them. The 448 px arm is optional (about 4x slower).

In [ ]:
!python scripts/e6_r3_ordinal.py --workers 4

In [ ]:
# optional: ordinal head at 448 px
!python scripts/e6_r3_ordinal.py --variants dinov3_b_ord448 --workers 4

## E3: quantization

In [ ]:
!python scripts/e3a_weight_sweep.py --workers 4

In [ ]:
!python scripts/e3b_onnx_int8.py   # CPU; latency is measured on this VM's CPU (recorded in cost.json)

## E4: binary models (third-party repos and BHViT checkpoints)

In [ ]:
!mkdir -p external/ckpt/bhvit_tiny
![ -d external/BinaryAttention ] || git clone -q --depth 1 https://github.com/EdwardChasel/BinaryAttention.git external/BinaryAttention
![ -d external/BHViT ] || git clone -q --depth 1 https://github.com/IMRL/BHViT.git external/BHViT
![ -f external/ckpt/bhvit_tiny/BHViT-tiny/best64.0.pth ] || (cd external/ckpt/bhvit_tiny && gdown -q --folder https://drive.google.com/drive/folders/1tuEdd8xkLSuwoordYdEl4VpKLRmP3xJO)
!ls -la external/ckpt/bhvit_tiny/BHViT-tiny

In [ ]:
!python scripts/e4_binary.py binatt --workers 4

In [ ]:
!python scripts/e4_binary.py bhvit_fp --workers 4

In [ ]:
!python scripts/e4_binary.py bhvit_bin --workers 4

## Cost, E5 and all tables and figures

In [ ]:
!python scripts/cost_fp.py
!python scripts/analyze.py
!cat analysis/table1.md analysis/table2.md

In [ ]:
from IPython.display import Image, display
display(Image('analysis/fig1_auroc_vs_bits.png')); display(Image('analysis/fig2_missed_vs_bits.png'))

## Explainability (post hoc, after E2/E3a)
Grad-CAM of p(TZ3), deletion faithfulness, and map stability across weight bit widths, written to `explain/` on Drive. Finished models are skipped on rerun.

In [ ]:
!python scripts/explain.py --split valB --workers 4
display(Image('explain/fig_stability.png')); display(Image('explain/panels/models.png'))

## Files for the paper (Overleaf)
Download `paper_files.zip` from the repo folder on Drive and upload its contents next to `paper.tex` in Overleaf.

In [ ]:
!python scripts/collect_paper_files.py